# GraphRAG vs Baseline Text-to-SQL — Evaluation Pipeline (Kaggle)

Notebook ini menjalankan **dua pipeline Text-to-SQL** pada Spider dev set, lalu mengevaluasinya dengan lapisan metrik skripsi. Semua logika ada di `src/` — notebook ini memanggilnya langkah demi langkah dan **menjelaskan apa yang terjadi di tiap langkah**, jadi bisa dibaca sebagai dokumentasi cara kerja pipeline.

## Dua pipeline yang dibandingkan

| Tahap | GraphRAG (`pipeline.py` → `run_single`) | Baseline (`baseline.py` → `run_single_baseline`) |
|---|---|---|
| Input | pertanyaan | pertanyaan |
| Pilih tabel | Stage 1: **tabel** dengan skor ≥ threshold tabel (n-gram + cosine BGE-M3) **sama** dengan Stage 1 GraphRAG (n-gram + cosine, skor ≥ threshold tabel), tapi dicocokkan ke teks tabel `table <nama> columns <kolom...>` |
| Pilih kolom | Stage 2: **kolom** di tabel kandidat dengan skor ≥ threshold kolom, + kolom dari **value linking** (nilai sel yang disebut di pertanyaan) | — (tidak ada) |
| Jalur JOIN | Graph traversal: shortest path antar **kolom**; yang masuk prompt hanya **kolom JOIN** (`fk_only`) | Table path tracing: shortest path antar **tabel** via FK |
| Schema context | DDL hanya **kolom terpilih** | DDL **semua kolom** tabel terpilih (tanpa pruning) |
| Few-shot | top-k contoh dari `train_spider.json`, dipilih by **skeleton** pertanyaan | **sama persis** (`cfg.few_shot_k`, index yang sama) |
| Prompt → LLM → cleaner | Prompt builder → Qwen2.5-Coder (4-bit, greedy) → SQL cleaner | **sama persis** untuk keduanya |

Kedua pipeline memakai LLM dan few-shot yang **sama**; yang berbeda hanya retrieval-nya.

GraphRAG memakai **preprocessing tetap** hasil trial SLA (`notebooks/graphrag_sla_trial.ipynb`, trial G7); yang bisa dipilih hanya threshold tabel dan kolom (sel 3). Few-shot dipilih dengan **skeleton** pertanyaan (`src/skeleton/`, `few_shot_selection="skeleton"`).

## Tiga tahap ablation (keputusan `IMPLEMENTATION_DECISIONS.md` poin 28)

| Tahap | Yang dicari | Grid (`config.py`) | Dipilih dengan | Jalankan |
|---|---|---|---|---|
| 1 | threshold tabel & kolom (dipakai kedua pipeline untuk tabel; kolom hanya GraphRAG) | grid di `graphrag_sla_trial.ipynb` | **SLA** (tanpa LLM) | `graphrag_sla_trial.ipynb` |
| 2 | few-shot k (sama untuk kedua pipeline) | `few_shot_frame` | **EX** | `ablation.py --mode {graphrag,baseline}` |
| 3 | model LLM | `llm_model_frame` | EX (+ token) | `MODEL` di Bagian 3, atau `pipeline.py --baseline --models all` |

Notebook ini menjalankan **satu** konfigurasi (satu model) dari ujung ke ujung; tahap 2 dan 3 lebih praktis lewat CLI karena butuh banyak run LLM.

## Alur notebook ini

| Bagian | Isi |
|---|---|
| 1-2 | Setup dan import |
| 3 | **Pilih `MODEL`**: satu-satunya yang diganti tiap orang |
| 4-8 | Baca data, bangun graph, muat model, buat index, siapkan evaluator |
| 9-10 | Contoh proses satu pertanyaan (boleh dilewati) |
| 11-12 | Jalankan 1-Stage dan 2-Stage, lalu amankan prediksi dan `raw_logs` |
| 13 | **Hasil 6 metrik**: satu tabel ringkasan, lalu rincian per metrik |
| 14 | Pembanding: evaluator resmi Spider |
| 15 | **Analisis 6 dimensi** |
| 16 | **Simpan semua hasil** ke JSON dan Excel |

Logika detail ada di `src/experiments/notebook_steps.py`, jadi sel di notebook ini tetap pendek. **1-Stage** adalah `baseline` di kode, dan **2-Stage** adalah `graphrag` di kode.

## Cara memakai untuk tiga model

Tiap orang hanya mengganti `MODEL` di sel Bagian 3 (`"3B"`, `"7B"`, atau `"14B"`), lalu menjalankan semua sel. Notebook selalu menjalankan **1-Stage** (`baseline` di kode) dan **2-Stage** (`graphrag` di kode). Hasil tiap model masuk ke folder sendiri, lalu digabung dengan `combine_models.py` (Bagian 17).

Semua hasil ditampilkan sebagai tabel angka (`show()` dari `src/experiments/result_tables.py`, modul yang sama dengan export Excel). Notebook ini **tidak menulis interpretasi atau kesimpulan**; pembacaan hasil dilakukan manual.

> **Catatan run final:** untuk skripsi, set `SAMPLE_RATIO = 1.0` (full 1034 query). Data QVT (`build_qvt_variations.py`) menolak `raw_logs` hasil subsampling, karena `query_id` = posisi baris di `dev.json`.

In [ ]:
# ============================================================
# 1. SETUP — clone repo, install dependency, cek evaluator Spider
# Sel ini aman dijalankan ulang (semua langkah dicek dulu).
# ============================================================
# os untuk pindah folder, subprocess untuk menjalankan perintah terminal
import os, subprocess
# Path untuk menulis lokasi file dengan aman di semua sistem operasi
from pathlib import Path

# Alamat repo yang di-clone saat notebook jalan di Kaggle
REPO_URL = "https://github.com/william-darma-wijaya/thesis-2803.git"
# Branch repo yang dipakai
REPO_BRANCH = "feat/three-model-eval"

# Fungsi pembantu untuk menjalankan satu perintah terminal
def sh(cmd: str) -> None:
    # Tampilkan perintahnya supaya terlihat apa yang sedang dijalankan
    print(f"$ {cmd}")
    # Jalankan perintah; check=True membuat notebook berhenti kalau perintahnya gagal
    subprocess.run(cmd, shell=True, check=True)

# Fungsi pembantu untuk mengecek apakah sebuah folder adalah root repo ini
def is_repo_root(p: Path) -> bool:
    # Cek file milik repo ini, BUKAN cuma folder "src": Kaggle punya /kaggle/src sendiri,
    # jadi cek "../src" dari /kaggle/working salah mengira /kaggle sebagai root repo.
    return (p / "src" / "core" / "config.py").is_file()

# 1) cwd HARUS root repo: pipeline memakai path relatif (outputs/, data/raw_logs/, external/spider_eval/)
# Kalau folder sekarang bukan root repo, cari atau siapkan dulu
if not is_repo_root(Path.cwd()):
    # Kasus pertama: folder induknya adalah root repo
    if is_repo_root(Path.cwd().parent):  # notebook dijalankan dari notebooks/
        # Naik satu folder ke root repo
        os.chdir("..")
    # Kasus kedua: repo berada di subfolder atau belum ada
    else:                                # Kaggle: repo belum ada di /kaggle/working
        # Repo belum pernah di-clone
        if not is_repo_root(Path("thesis-2803").resolve()):
            # Clone repo dari GitHub
            sh(f"git clone -b {REPO_BRANCH} {REPO_URL}")
        # Repo sudah ada dari run sebelumnya
        else:                                # sudah pernah di-clone: pastikan di branch yang benar
            # Ambil versi terbaru dari branch yang dipilih
            sh(f"git -C thesis-2803 fetch origin {REPO_BRANCH} && git -C thesis-2803 checkout {REPO_BRANCH} && git -C thesis-2803 pull origin {REPO_BRANCH}")
        # Masuk ke folder repo
        os.chdir("thesis-2803")
# Simpan lokasi root repo untuk dipakai sel berikutnya
REPO_ROOT = Path.cwd()
# Hentikan notebook kalau ternyata masih bukan root repo
assert is_repo_root(REPO_ROOT), f"{REPO_ROOT} bukan root repo thesis-2803 (src/core/config.py tidak ada)"
# Tampilkan lokasi repo
print("Repo root:", REPO_ROOT)
# Tampilkan branch yang sedang aktif
print("Branch   :", subprocess.run("git rev-parse --abbrev-ref HEAD", shell=True, capture_output=True, text=True).stdout.strip())

# 2) dependency (torch, pandas, networkx, tqdm sudah ada di Kaggle -> jangan di-upgrade;
#    `-U pandas` sebelumnya menarik pandas 3.x dan bentrok dengan dependency image Kaggle).
#    nltk dibutuhkan process_sql.py milik Spider.
# Install library yang belum tersedia di Kaggle
sh("pip install -q -U 'bitsandbytes>=0.46.1' sentence-transformers transformers nltk")

# 3) evaluator resmi Spider sudah ter-commit di external/spider_eval/ (JANGAN diedit) -> cukup dicek ada
# Periksa kedua file evaluator resmi satu per satu
for fname in ("evaluation.py", "process_sql.py"):
    # Hentikan notebook kalau filenya tidak ada
    assert (REPO_ROOT / "external" / "spider_eval" / fname).exists(), f"external/spider_eval/{fname} tidak ada di repo"

# 4) tokenizer data untuk parser SQL Spider
# nltk dipakai parser Spider untuk memecah SQL menjadi token
import nltk
# Unduh dua paket data tokenizer yang dibutuhkan
for pkg in ("punkt", "punkt_tab"):
    # Coba unduh paketnya
    try:
        # quiet=True supaya tidak banyak output
        nltk.download(pkg, quiet=True)
    # Kalau gagal, beri peringatan tanpa menghentikan notebook
    except Exception as e:
        # Tampilkan paket yang gagal dan alasannya
        print(f"[WARN] nltk.download({pkg}) gagal: {e}")
# Tanda bahwa seluruh setup sudah selesai
print("Setup selesai.")

## 2. Import

Semua langkah notebook ada di `src/experiments/notebook_steps.py` dan dipanggil lewat `steps`, misalnya `steps.load_models(cfg)`. Isi file itu juga diberi penjelasan per baris.

In [ ]:
# sys dipakai untuk mendaftarkan lokasi kode
import sys
# Daftarkan root repo supaya modul src bisa di-import
sys.path.insert(0, str(REPO_ROOT))

# Konfigurasi utama pipeline
from src.core.config import PipelineConfig
# Menentukan folder hasil per model, dan menjalankan analisis metrik serta dimensi
from src.experiments.pipeline import model_run_dirs, run_metrics_analysis, compute_dimension_analysis
# Fungsi pembuat tabel hasil
from src.experiments import result_tables as rt
# Fungsi untuk menampilkan tabel di notebook
from src.experiments.result_tables import show
# Semua langkah notebook, supaya sel di bawah tetap pendek
from src.experiments import notebook_steps as steps

# Pastikan semua import berhasil
print("Imports OK")

## 3. Konfigurasi

**Yang diganti tiap orang hanya `MODEL`**: `"3B"`, `"7B"`, atau `"14B"`. Hasil tiap model masuk ke `outputs/models/<model>/`, jadi tidak saling menimpa.

| Pengaturan | Arti |
|---|---|
| `SAMPLE_RATIO` | `0.05` untuk percobaan cepat (sekitar 51 pertanyaan, hasil di `outputs/trial/models/<model>/`), `1.0` untuk run final |
| `TABLE_THRESHOLD`, `COLUMN_THRESHOLD` | Skor minimum tabel dan kolom, dipilih di `graphrag_sla_trial.ipynb` |
| `FEW_SHOT_SELECTION` | Cara memilih contoh few-shot, berlaku untuk kedua varian |

Pengaturan lain ada di `src/core/config.py` dan jangan diubah di sini.

In [ ]:
# ===== BAGIAN YANG DIGANTI TIAP ORANG: hanya MODEL =====
# Model LLM yang dijalankan: "3B", "7B", atau "14B"
MODEL = "7B"
# Porsi dev set yang dipakai: 0.05 = percobaan cepat, 1.0 = run final skripsi
SAMPLE_RATIO = 0.05

# Skor minimum sebuah tabel supaya terpilih (Stage 1)
TABLE_THRESHOLD = 0.60
# Skor minimum sebuah kolom supaya terpilih (Stage 2)
COLUMN_THRESHOLD = 0.70
# Cara memilih contoh few-shot: "question" atau "skeleton"
FEW_SHOT_SELECTION = "skeleton"

# Buat konfigurasi dengan pilihan di atas
cfg = PipelineConfig(table_similarity_threshold=TABLE_THRESHOLD,
                     # Threshold kolom
                     semantic_similarity_threshold=COLUMN_THRESHOLD,
                     # Cara memilih few-shot
                     few_shot_selection=FEW_SHOT_SELECTION)
# Ubah nama pendek ("14B") menjadi nama lengkap model
cfg.llm_model = cfg.resolve_llm_model(MODEL)
# Hentikan notebook kalau dataset Spider tidak ditemukan
assert cfg.data_path.exists(), f"Data Spider tidak ditemukan di {cfg.data_path}. Di Kaggle, tambahkan lewat 'Add Input'."

# True kalau ini run penuh, False kalau run percobaan
IS_FULL = SAMPLE_RATIO >= 1.0
# Tentukan folder hasil, raw_logs, dan QVT khusus untuk model ini
ARTIFACT_DIR, RAW_LOGS_DIR, QVT_DIR = model_run_dirs(cfg.llm_model, out_root=Path("outputs") if IS_FULL else Path("outputs/trial"))
# Run percobaan menyimpan raw_logs di folder trial supaya tidak menimpa run penuh
if not IS_FULL:
    # Arahkan raw_logs ke dalam folder hasil trial
    RAW_LOGS_DIR = ARTIFACT_DIR / "raw_logs"
# Buat folder prediksi, tabel, dan raw_logs
for folder in (ARTIFACT_DIR / "predictions", ARTIFACT_DIR / "tables", RAW_LOGS_DIR):
    # Tidak error kalau foldernya sudah ada
    folder.mkdir(parents=True, exist_ok=True)

# Kunci semua sumber acak supaya hasil run bisa diulang
steps.set_seed(cfg.seed)
# Tampilkan konfigurasi run
steps.show_run_config(cfg, SAMPLE_RATIO, ARTIFACT_DIR, RAW_LOGS_DIR)

## 4. Baca Data

`dev.json` berisi 1034 pasang pertanyaan dan gold SQL. Run percobaan mengambil sampel dengan seed tetap, jadi sampelnya selalu sama.

In [ ]:
# Baca pertanyaan dev set; run percobaan hanya mengambil sebagian
questions = steps.load_dev_questions(cfg, SAMPLE_RATIO)
# Tulis gold SQL sesuai sampel, untuk evaluator resmi di Bagian 14
gold_path = steps.write_gold_file(questions, ARTIFACT_DIR)
# Tampilkan contoh pertanyaan pertama
print("Contoh:", questions[0]["question"])

## 5. Bangun Schema Graph

- **2-Stage:** satu node mewakili satu kolom, jadi seleksi bisa sampai level kolom.
- **1-Stage:** satu node mewakili satu tabel, jadi semua kolom tabel terpilih masuk prompt.

In [ ]:
# Bangun graph per kolom (2-Stage) dan graph per tabel (1-Stage)
col_graph, tbl_graph = steps.build_graphs(cfg)

## 6. Muat Model

Dua model dimuat sekali dan dipakai kedua varian: `BAAI/bge-m3` untuk embedding, dan LLM sesuai `MODEL` (4-bit, greedy). Pastikan Accelerator = GPU.

In [ ]:
# Muat model embedding dan LLM sesuai MODEL
embed_model, llm, tokenizer = steps.load_models(cfg)

## 7. Buat Index

Embedding schema dan contoh few-shot dibuat sekali di memori. Contoh few-shot hanya diambil dari `train_spider.json`, jadi tidak ada kebocoran data dev.

In [ ]:
# Buat embedding schema dan index contoh few-shot
col_cache, tbl_cache, few_shot_idx = steps.build_indexes(cfg, questions, col_graph, tbl_graph, embed_model)

## 8. Siapkan Evaluator

ESM, EX, dan CM per pertanyaan dihitung oleh `src/metrics/esm_ex_cm.py` (kode custom yang membungkus parser dan evaluator resmi Spider).

In [ ]:
# Siapkan alat untuk menghitung ESM, EX, CM, dan kolom gold
evaluator = steps.SqlEvaluator(cfg)
# Satukan semua alat supaya mudah diberikan ke fungsi berikutnya
tools = steps.Tools(cfg, col_graph, tbl_graph, embed_model, llm, tokenizer, col_cache, tbl_cache, few_shot_idx, evaluator)

## 9. Contoh Satu Pertanyaan: 2-Stage

Menampilkan tabel, kolom, DDL, contoh few-shot, dan SQL untuk satu pertanyaan. Ini hanya contoh dan **tidak masuk hasil akhir**, jadi boleh dilewati.

In [ ]:
# Posisi pertanyaan yang ingin dilihat; ganti untuk melihat pertanyaan lain
WALK_IDX = 0
# Tampilkan proses 2-Stage untuk pertanyaan itu
two_stage_example = steps.walk_through_2stage(tools, questions[WALK_IDX])

## 10. Contoh Satu Pertanyaan: 1-Stage

Pertanyaan yang sama lewat 1-Stage: yang dipilih tabel utuh, dan semua kolomnya masuk prompt. Contoh few-shot-nya sama dengan 2-Stage.

In [ ]:
# Tampilkan proses 1-Stage untuk pertanyaan yang sama, lalu bandingkan
steps.walk_through_1stage(tools, questions[WALK_IDX], two_stage_example)

## 11. Jalankan Kedua Varian

Untuk tiap pertanyaan: jalankan 2-Stage dan 1-Stage, evaluasi kedua SQL, lalu simpan satu catatan per varian.

Kalau generate gagal, SQL diganti `SELECT 1` dan run tetap lanjut. Kalau evaluasi gagal, prediksi tetap tersimpan. Jumlah error ditampilkan di akhir, dan untuk run final sebaiknya 0.

In [ ]:
# Jalankan 1-Stage dan 2-Stage untuk semua pertanyaan
results = steps.run_all_questions(tools, questions)
# Tampilkan jumlah pertanyaan yang diproses dan jumlah error
steps.show_loop_summary(results)

## 12. Amankan Prediksi dan raw_logs

File ini sengaja disimpan sekarang, sebelum hasil lain. Metrik dan dimensi membacanya, dan hasil generate yang memakan waktu lama tidak hilang kalau langkah berikutnya gagal. Hasil lain baru disimpan di Bagian 16.

In [ ]:
# Simpan prediksi dan raw_logs supaya hasil generate tidak hilang kalau langkah berikutnya gagal
steps.save_generation_results(results, ARTIFACT_DIR, RAW_LOGS_DIR)

## 13. Hasil 6 Metrik

Tabel pertama merangkum **SLA, ESM, EX, CM, Token + TEP, dan QVT** dalam satu tabel. CM ditampilkan per klausa karena tidak boleh digabung menjadi satu angka. QVT hanya terisi pada run penuh (`SAMPLE_RATIO = 1.0`).

Selisih persentase ditulis dalam poin persen (pp). Hasil di bagian ini belum disimpan.

In [ ]:
# Hitung keenam metrik dari raw_logs; hasilnya belum disimpan
metrics_results = run_metrics_analysis(RAW_LOGS_DIR, ARTIFACT_DIR / "tables", cfg.dev_json, IS_FULL, cfg, SAMPLE_RATIO, QVT_DIR, save=False)
# Tampilkan keenam metrik dalam satu tabel
show(rt.metrics_summary_table(metrics_results))

### 13.1 Schema Linking Accuracy (SLA)

Precision, recall, dan F1 skema yang dikirim ke LLM dibandingkan dengan skema gold SQL. Nilainya dirata-ratakan per query.

In [ ]:
# Tampilkan rincian SLA
steps.show_metric(metrics_results, "M1 SLA")

### 13.2 Exact Set Match (ESM)

Persentase prediksi yang struktur SQL-nya sama dengan gold.

In [ ]:
# Tampilkan rincian ESM per tingkat kesulitan
steps.show_metric(metrics_results, "M2 ESM")

### 13.3 Execution Accuracy (EX)

Persentase prediksi yang hasil eksekusinya sama dengan gold.

In [ ]:
# Tampilkan rincian EX per tingkat kesulitan
steps.show_metric(metrics_results, "M3 EX")

### 13.4 Component Match (CM)

Kecocokan per klausa SQL. Nilainya sengaja tidak digabung menjadi satu angka.

In [ ]:
# Tampilkan rincian CM per klausa
steps.show_metric(metrics_results, "M4 CM")

### 13.5 Token Consumption dan TEP

T = T_in + α × T_out. TEP = (ΔEX / EX 1-Stage) / (ΔT / T 1-Stage), dengan Δ = 2-Stage − 1-Stage.

In [ ]:
# Tampilkan rincian Token dan TEP
steps.show_metric(metrics_results, "M5 Token TEP")

### 13.6 Query Variance Testing (QVT)

Kestabilan jawaban pada pasangan pertanyaan parafrase. Grup yang semua variasinya salah tidak dihitung.

In [ ]:
# Tampilkan rincian QVT
steps.show_metric(metrics_results, "M6 QVT")

### 13.7 Tabel Pendukung

Rincian tambahan untuk membaca metrik. Tabel di sini memakai nama kode: Baseline = 1-Stage, GraphRAG = 2-Stage.

In [ ]:
# Tampilkan tabel pendukung untuk membaca metrik
steps.show_diagnostics(results, cfg.token_output_weight)

## 14. Pembanding: Evaluator Resmi Spider

Menjalankan `external/spider_eval/evaluation.py` yang tidak diubah. Angka utama tetap dari Bagian 13, karena evaluator resmi gagal mem-parse outer JOIN (`IMPLEMENTATION_DECISIONS.md` poin 11).

In [ ]:
# Jalankan evaluator resmi Spider sebagai pembanding; hasilnya belum disimpan
spider_scores = steps.run_spider_check(cfg, ARTIFACT_DIR, gold_path)

## 15. Analisis 6 Dimensi

| Dimensi | Pertanyaan yang dijawab |
|---|---|
| **1. Efisiensi** | Apakah lebih hemat token tanpa mengorbankan akurasi? |
| **2. Struktur SQL** | Bagaimana EX dan ESM berubah bersama? |
| **3. Komponen SQL** | Klausa mana yang paling terdampak? |
| **4. Robustness** | Apakah jawaban konsisten saat pertanyaan diparafrase? |
| **5. Bottleneck** | Kalau akurasi rendah, gagal di retrieval atau generation? |
| **6. Few-shot** | Berapa k yang cukup? |

- **Dimensi 4** hanya jalan pada run penuh.
- **Dimensi 5** hanya jalan kalau Dimensi 2 menemukan EX dan ESM sama-sama rendah. `ALWAYS_DIM5 = True` memaksanya jalan.
- **Dimensi 6** butuh hasil `ablation.py`. Ablation cukup dijalankan dengan satu model, jadi angkanya sama di ketiga model.

Dimensi yang datanya belum ada dilewati dengan keterangan. Hasil di bagian ini belum disimpan.

In [ ]:
# Baca EX per k dari hasil ablation.py untuk Dimensi 6
EX_PER_K = steps.read_ex_per_k(cfg)
# False berarti Dimensi 5 mengikuti hasil Dimensi 2
ALWAYS_DIM5 = False
# Hitung Dimensi 1-6; hasilnya belum disimpan, dan data QVT sudah dibuat di Bagian 13
dim_results, dim_report = compute_dimension_analysis(RAW_LOGS_DIR, cfg.dev_json, IS_FULL, EX_PER_K, ALWAYS_DIM5, QVT_DIR, build_qvt=False)
# Tampilkan status dan tabel tiap dimensi
steps.show_dimensions(dim_results)

## 16. Simpan Semua Hasil

Setelah semua hasil tampil di atas, sel ini menyimpan semuanya ke folder model:

| File | Isi |
|---|---|
| `tables/metrics_results.json` | Keenam metrik |
| `tables/dimensions_results.json` | Hasil dimensi |
| `tables/hasil_skripsi.xlsx` | Ringkasan, rincian metrik, dimensi, dan tabel pendukung |
| `tables/spider_official_eval.csv` | Skor evaluator resmi |
| `tables/comparison_report.txt`, `tables/confound_diagnostics.json` | Proxy dan confound diagnostics |

Di Kaggle, sel ini juga membuat `hasil_skripsi_<MODEL>.xlsx` dan `hasil_<MODEL>.zip` di `/kaggle/working/` untuk diunduh.

Untuk menjalankan ulang dimensi saja tanpa generate ulang, contoh untuk 14B:

```bash
python src/experiments/run_all_dimensions.py --raw-logs-dir data/raw_logs/Qwen2.5-Coder-14B-Instruct --qvt-dir data/qvt_variations/Qwen2.5-Coder-14B-Instruct --out-dir outputs/models/Qwen2.5-Coder-14B-Instruct/tables --ex-per-k-file outputs/tables/ablation_ex_per_k_graphrag_skeleton.json
```

## 17. Menggabungkan Ketiga Model

Setelah 3B, 7B, dan 14B selesai, ekstrak zip dari tiap orang ke `outputs/models/`, lalu jalankan:

```bash
python src/experiments/combine_models.py
```

Hasilnya `outputs/tables/hasil_gabungan.xlsx` dan `hasil_gabungan.json`. Skrip ini memberi peringatan kalau ada model yang belum ada, run percobaan, atau konfigurasi yang berbeda antar model.

In [ ]:
# Simpan semua hasil ke JSON dan Excel, setelah semuanya ditampilkan di atas
xlsx_path = steps.save_all_results(MODEL, results, metrics_results, spider_scores, dim_results, dim_report, ARTIFACT_DIR, RAW_LOGS_DIR)

## Catatan Desain yang Perlu Disebut di Metodologi

- **Few-shot sama untuk kedua pipeline.** Baseline dan GraphRAG sama-sama memakai `cfg.few_shot_k` contoh dari index yang sama (`build_few_shot_block()`), jadi selisih GraphRAG–Baseline murni dari retrieval. k final dipilih di ablation tahap 2 (EX tertinggi, rata-rata kedua pipeline).
- **Seleksi GraphRAG berbasis threshold**: tabel dan kolom disaring dengan skor cosine minimum, tanpa batas top-k%; threshold dipilih dengan SLA di `graphrag_sla_trial.ipynb`. Langkah lain (nama natural, stopword `edge`, value linking teks, `fk_only`, sambung tabel tanpa PK) adalah **preprocessing tetap**, kontribusi masing-masing ditunjukkan rekap trial G1-G7 di notebook itu. Baseline memilih tabel dengan cara yang sama (n-gram, threshold tabel, tanpa top-k%), hanya tanpa tahap kolom: seluruh kolom tabel terpilih dikirim.
- **Traversal `fk_only`.** Dari jalur traversal hanya kolom JOIN yang masuk prompt; tabel tanpa PK disambung di graph traversal supaya JOIN lewat tabel penghubung terbaca.
- **Nama natural Spider** (`embed_natural_names`) dipakai sebagai teks embed GraphRAG **dan** Baseline (preprocessing data schema yang sama, jadi selisih hasil murni dari cara retrieval); anotasinya berasal dari `tables.json` Spider sendiri. SQL dan DDL tetap nama asli.
- **Few-shot skeleton** (`few_shot_selection="skeleton"`): contoh dipilih dengan kemiripan pertanyaan yang domain-nya di-mask (de-semanticization, Guo et al., 2023, disederhanakan); prompt tetap berisi pertanyaan + SQL asli contohnya. Sama untuk kedua pipeline.
- **Few-shot hanya dari `train_spider.json`.** Retrieval dari train set, bukan kebocoran dev/test.
- **1-Stage dan 2-Stage.** `baseline` di kode adalah GraphRAG 1-Stage, dan `graphrag` di kode adalah GraphRAG 2-Stage. Nama lama masih dipakai untuk file dan kunci data.
- **Proxy vs SLA.** Recall/Precision yang diprint pipeline adalah proxy name-matching, bukan SLA; SLA resmi dari `src/metrics/sla.py`.
- **ESM ≡ EM di Spider; EX order-sensitive** (`eval_exec_match()` resmi apa adanya). Diagnostik confound di Bagian 13.7 menghitung kasusnya tanpa mengubah metrik.